# ADHD EEG Classification: CNN-LSTM vs CNN-LSTM-Attention

Training notebook built on **`ADHD_CLEAN_DATASET`** (`data/harmonized`, `data/cleaned`).
The cleaning pipeline is used **as-is**; nothing here recreates or modifies it, and
no raw data is modified.

**Dataset used for training**: BALLADEER Emotiv EPOC EEG (declared `eeg_128` Hz subset).
`adhdata` (19-ch) and BALLADEER CGX are inspected but **excluded from band extraction**
because their sampling rate cannot be determined from the source (see Section 1).

> **Data-integrity rules honoured throughout**
> 1. Never invent missing data. 2. Never assume sampling rate. 3. Never fabricate timestamps.
> 4. Never fabricate diagnosis labels. 5. Never randomly split EEG windows.
> 6. Never allow participant leakage. 7. Never fit normalization on the whole dataset.
> 8. Never use the test set for tuning. 9. Gamma is an **extension** of the baseline, not part of it.
> 10. Attention weights do **not** prove biological importance.
> 11. Window-level metrics are never reported as subject-level metrics.

## 1. Dataset inspection (cleaned dataset only)

In [ ]:
import os, sys, json, re, random, time, copy, ntpath, glob
from datetime import datetime
import numpy as np
import pandas as pd
import joblib

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# -------------------- Google Drive / Colab --------------------
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

MYDRIVE = '/content/drive/MyDrive'

def find_named_dir(root, dirname, max_hits=20):
    hits = []
    for base, dirs, files in os.walk(root):
        if os.path.basename(base).lower() == dirname.lower():
            hits.append(base)
            if len(hits) >= max_hits:
                break
    return hits

project_candidates = [os.path.join(MYDRIVE, 'ADHD_CLEAN_DATASET')]
project_candidates += find_named_dir(MYDRIVE, 'ADHD_CLEAN_DATASET')
project_candidates = list(dict.fromkeys(project_candidates))
PROJECT = next((p for p in project_candidates if os.path.isdir(p)), None)
if PROJECT is None:
    raise FileNotFoundError(
        'ADHD_CLEAN_DATASET was not found in MyDrive. '
        'Upload/mount that folder first.'
    )

HARM  = os.path.join(PROJECT, 'data', 'harmonized')
CLEAN = os.path.join(PROJECT, 'data', 'cleaned')

paths = {
    'eeg_subject_index': os.path.join(HARM, 'EEG_ONLY', 'eeg_subject_index.csv'),
    'eeg_labels':        os.path.join(HARM, 'EEG_ONLY', 'eeg_labels.csv'),
    'participant_meta':  os.path.join(HARM, 'participant_metadata.csv'),
    'dataset_registry':  os.path.join(HARM, 'dataset_registry.csv'),
    'loso_grouping':     os.path.join(HARM, 'loso_grouping.csv'),
    'adhdata_parquet':   os.path.join(CLEAN, 'eeg', 'adhdata_clean.parquet'),
    'epoc_sessions':     os.path.join(CLEAN, 'eeg', 'balladeer_eeg_sessions.csv'),
}

print('PROJECT:', PROJECT)
for k, fp in paths.items():
    print(f'{k:22s} exists={os.path.exists(fp)}  {fp}')

# Optional raw BALLADEER directory. The cleaned metadata contains old Windows
# paths, so the actual raw CSVs must be resolved on the mounted Drive.
balladeer_candidates = find_named_dir(MYDRIVE, 'BALLADEER ADHD DATASET')
RAW_BALLADEER = balladeer_candidates[0] if balladeer_candidates else None
print('\nRAW_BALLADEER:', RAW_BALLADEER)
if RAW_BALLADEER is None:
    print('WARNING: raw BALLADEER directory was not auto-found. The EPOC feature cell will report missing files.')


In [ ]:
eeg_idx = pd.read_csv(paths['eeg_subject_index'])
labels  = pd.read_csv(paths['eeg_labels'])
pmeta   = pd.read_csv(paths['participant_meta'])
registry = pd.read_csv(paths['dataset_registry'])

# Normalize diagnosis strings once, without changing the underlying source files.
labels['diagnosis'] = labels['diagnosis'].astype(str).str.strip()
pmeta['diagnosis'] = pmeta['diagnosis'].astype(str).str.strip()

print('---------------- adjusted diagnoses: participant_metadata ----------------')
print(pmeta['diagnosis'].value_counts(dropna=False).to_string())
print('\n---------------- EEG_ONLY subject labels ----------------')
print(labels['diagnosis'].value_counts(dropna=False).to_string())
print('\nEEG_ONLY sessions:', eeg_idx['session_id'].nunique(),
      '| subjects:', eeg_idx['participant_id'].nunique())

print('\n---------------- sessions by kind ----------------')
print(eeg_idx.groupby(['kind', 'device']).agg(
    n_sessions=('session_id', 'nunique'),
    n_subjects=('participant_id', 'nunique')
).to_string())

print('\n---------------- session tasks (BALLADEER) ----------------')
print(eeg_idx[eeg_idx['source_dataset'] == 'balladeer']['task'].value_counts().to_string())

print('\n---------------- per-declared sampling rate (EPOC files) ----------------')
epoc = eeg_idx[eeg_idx['kind'].astype(str).str.startswith('epoc')].copy()
print(epoc['sampling_rate'].value_counts(dropna=False).to_string())

print('\n---------------- EPOC channels (first sessions) ----------------')
print(epoc['channel_list'].head(3).to_string())


**Sampling-rate availability (checked in cleaned metadata, never assumed)**

| EEG source | Sampling rate status | Consequence |
|---|---|---|
| `adhdata_clean.parquet` | **Sampling rate unavailable** (not stated in file; no timestamps) | excluded from band extraction — per integrity rule |
| BALLADEER CGX (Slackline) | only a **computed estimate** (`~500 Hz`, unverified device rate) | excluded from band extraction — estimate not treated as determined |
| BALLADEER Emotiv EPOC | **declared in each file** (`eeg_128;mot_32;pm_0.1;pow_8` -> 128 Hz EEG, `eeg_256;...` -> 256 Hz) | **used for training** (128 Hz subset for rate consistency) |

All EPOC files that declare `eeg_128` are kept; the 7 files declaring `eeg_256` are
excluded from the training dataset for a **consistent sampling rate** (documented).

In [ ]:
# --- verify the declared rate string and filter to the 128 Hz EEG subset ---
def extract_eeg_hz(value):
    m = re.search(r'eeg_(\d+)', str(value))
    return int(m.group(1)) if m else np.nan

epoc['declared_eeg_hz'] = epoc['sampling_rate'].map(extract_eeg_hz)
print(epoc.groupby('declared_eeg_hz')['session_id'].count().to_string())

EPOC_SESSIONS = epoc[epoc['declared_eeg_hz'] == 128].reset_index(drop=True).copy()
EPOC_SR = 128

print(f'\n128 Hz EPOC sessions retained: {len(EPOC_SESSIONS)}')
print(f'subjects in 128 Hz subset: {EPOC_SESSIONS["participant_id"].nunique()}')
print('\nmissing values in eeg_subject_index:', int(eeg_idx.isna().sum().sum()))
print('duplicate (participant, session) rows:', int(eeg_idx.duplicated(['participant_id', 'session_id']).sum()))


## 2. Participant-level splitting (no leakage)

Every EEG window/sequence belongs to exactly one participant. **No random window-level
split is allowed.** Evaluation uses **`StratifiedGroupKFold` (5 folds)** with
`groups = participant_id`, so all data of one participant stays in one fold.

(If the participant count were small enough LOSO would be used; 123 participants make
LOSO impractical for CNN-LSTM training times, so grouped 5-fold is used. Every fold
preserves the ADHD/Control stratification at the participant level.)

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit

N_SPLITS = 5

print("evaluation strategy: StratifiedGroupKFold(n_splits=%d, shuffle=True, random_state=%d)"
      % (N_SPLITS, SEED))
print("groups = participant_id  -> participant leakage prevented by construction")

## 3. EEG preprocessing (validated loader)

In [ ]:
from scipy.signal import welch
from scipy.integrate import trapezoid
import warnings

EPOC_CHANNELS = ['AF3','F7','F3','FC5','T7','P7','O1','O2','P8','T8','FC6','F4','F8','AF4']

def read_epoc_session(file_path, expected_rate=128, channels=EPOC_CHANNELS):
    """Read one Emotiv EPOC export and return validated EEG samples."""
    if not file_path or not os.path.isfile(file_path):
        raise FileNotFoundError(f'EPOC file not found: {file_path}')

    with open(file_path, encoding='utf-8', errors='replace') as fh:
        meta_line = fh.readline()
        header_line = fh.readline()

    header_cols = [x.strip() for x in header_line.strip().split(',')]
    meta = {}
    for key in ['title', 'start timestamp', 'stop timestamp', 'headset type',
                'headset serial', 'sampling rate', 'samples']:
        m = re.search(re.escape(key) + r':\s*([^,]+)', meta_line)
        if m:
            meta[key] = m.group(1).strip()

    declared = meta.get('sampling rate', '')
    m_rate = re.search(r'eeg_(\d+)', declared)
    declared_rate = int(m_rate.group(1)) if m_rate else None
    if declared_rate != expected_rate:
        raise ValueError(f'declared EEG rate={declared_rate}, expected={expected_rate}')

    # The first line is metadata and the second line is the real CSV header.
    df = pd.read_csv(
        file_path, skiprows=2, header=None, names=header_cols,
        on_bad_lines='skip', engine='c'
    )

    col_need = [f'EEG.{c}' for c in channels]
    missing_cols = [c for c in col_need if c not in df.columns]
    if missing_cols:
        raise ValueError(f'missing EEG columns {missing_cols} in {file_path}')

    sig = df[col_need].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=np.float32)
    finite = np.isfinite(sig).all(axis=1)
    n_dropped = int((~finite).sum())

    eeg_cols = [c for c in df.columns if str(c).startswith('EEG.')]
    order_ok = [c for c in eeg_cols if c in col_need] == col_need

    # Timestamp is only used to verify ordering. It is never passed to the model.
    timestamp_ok = True
    timestamp_col = next((c for c in df.columns if str(c).strip().lower() == 'timestamp'), None)
    if timestamp_col is not None:
        ts = pd.to_numeric(df[timestamp_col], errors='coerce').to_numpy()
        ts = ts[np.isfinite(ts)]
        timestamp_ok = bool(len(ts) < 2 or np.all(np.diff(ts) >= 0))
        if not timestamp_ok:
            raise ValueError(f'timestamp ordering is not monotonic in {file_path}')

    return {
        'sig': sig.astype(np.float32),
        'valid': finite,
        'n_rows': sig.shape[0],
        'n_dropped': n_dropped,
        'declared_rate': declared_rate,
        'rate_ok': True,
        'order_ok': order_ok,
        'timestamp_ok': timestamp_ok,
        'channels': eeg_cols,
        'meta': meta,
    }


**Timestamps / ordering** — the raw `Timestamp` column is epoch-ms and is **not**
passed to the network. It is only used to (a) keep rows in chronological order
(they already are) and (b) detect ordering reversals. If a timestamp column exists in
a session, a monotonicity check is run during feature extraction and reported.

## 4. Time windows (baseline: 1 s)

In [ ]:
WINDOW_SEC = 1.0
SEQUENCE_LEN = 16          # timesteps (1-s windows) per sequence
SEQUENCE_STRIDE = SEQUENCE_LEN   # disjoint blocks -> no overlapping windows
SAMPLES_PER_WINDOW = int(round(EPOC_SR * WINDOW_SEC))   # NOT hard-coded
print("samples_per_window = int(rate * window_sec) =",
      EPOC_SR, "*", WINDOW_SEC, "=", SAMPLES_PER_WINDOW)

def contiguous_segments(valid_mask):
    """Indices of contiguous <run_start,run_end> blocks (no dropped rows inside)."""
    segs = []
    in_run = False
    for i, v in enumerate(valid_mask):
        if v and not in_run:
            start = i; in_run = True
        elif (not v) and in_run:
            segs.append((start, i)); in_run = False
    if in_run:
        segs.append((start, len(valid_mask)))
    return segs

def make_windows(sig, valid, fs, window_samples):
    """Non-overlapping 1-s windows from contiguous segments; incomplete tails dropped."""
    segs = contiguous_segments(valid)
    windows = []
    for a, b in segs:
        n_full = (b - a) // window_samples
        for k in range(n_full):
            windows.append(sig[a + k*window_samples: a + (k+1)*window_samples])
    return np.asarray(windows, dtype=np.float32) if windows else np.empty((0, window_samples, sig.shape[1]), dtype=np.float32)

## 5. Five frequency bands (window band-power features)

In [ ]:
# Gamma is an EXTENSION of the baseline (Delta/Theta/Alpha/Beta) - documented.
BANDS = {
    "Delta":  (0.5, 4.0),
    "Theta":  (4.0, 8.0),
    "Alpha":  (8.0, 13.0),
    "Beta":   (13.0, 30.0),
    "Gamma":  (30.0, 45.0),   # extension
}
BAND_NAMES = list(BANDS.keys())

nyquist = EPOC_SR / 2.0
print("Nyquist =", nyquist, "Hz")
for b, (lo, hi) in BANDS.items():
    ok = hi <= nyquist
    print(f"  {b:7s} {lo:5.1f}-{hi:4.1f} Hz  supported_by_128Hz={ok}")
assert all(hi <= nyquist for _, (lo, hi) in BANDS.items()), \
    "a band exceeds Nyquist for the declared 128 Hz rate - do not invent data"

def window_band_features(windows, fs, bands):
    """windows: (W, S, C) -> (W, C*len(bands)) band power features."""
    W, S, C = windows.shape
    power = np.empty((W, len(bands), C), dtype=np.float32)
    seg = windows if S == fs else np.pad(windows, ((0, 0), (0, fs - S), (0, 0)))
    # Welch on the full 1-s window (nperseg = window length)
    f, px = welch(seg, fs=fs, nperseg=fs, axis=1, scaling="spectrum")
    for bi, (bname, (lo, hi)) in enumerate(bands.items()):
        m = (f >= lo) & (f <= hi)
        power[:, bi, :] = trapezoid(px[:, m, :], x=f[m], axis=1)
    power = power.transpose(0, 2, 1)            # (W, C, B)
    return power.reshape(W, C * len(bands)).astype(np.float32)   # (W, 70)

Output per 1-s window: **14 channels x 5 bands = 70 features**.

## 6. Feature/tensor preparation & leakage-free normalization

In [ ]:
# ============================================================
# Resolve old Windows file paths -> actual Google Drive files
# ============================================================
CACHE_DIR = os.path.join(PROJECT, 'data', 'epoc_features_cache')
os.makedirs(CACHE_DIR, exist_ok=True)

# The metadata CSV stores Windows paths such as C:\\Users\\...\\file.csv.
# In Colab those paths are invalid. We match by the original filename and, when
# necessary, by the expected file size.

def build_raw_file_index(root):
    index = {}
    if root is None or not os.path.isdir(root):
        return index
    for base, dirs, files in os.walk(root):
        for fn in files:
            if fn.lower().endswith('.csv'):
                index.setdefault(fn, []).append(os.path.join(base, fn))
    return index

RAW_FILE_INDEX = build_raw_file_index(RAW_BALLADEER)
print('raw CSV files indexed:', sum(len(v) for v in RAW_FILE_INDEX.values()))
print('unique CSV filenames:', len(RAW_FILE_INDEX))


def windows_basename(path):
    # ntpath handles Windows separators even when running on Linux.
    return ntpath.basename(str(path).replace('/', '\\'))


def resolve_epoc_path(row):
    fn = str(row['file_name']) if pd.notna(row['file_name']) else windows_basename(row['file_path'])
    candidates = RAW_FILE_INDEX.get(fn, [])
    if not candidates:
        return None
    if len(candidates) == 1:
        return candidates[0]
    # If duplicate filenames exist, prefer the matching size from metadata.
    expected_size = row.get('size_bytes', np.nan)
    if pd.notna(expected_size):
        same_size = [p for p in candidates if os.path.getsize(p) == int(expected_size)]
        if len(same_size) == 1:
            return same_size[0]
    return candidates[0]

EPOC_SESSIONS['resolved_file_path'] = EPOC_SESSIONS.apply(resolve_epoc_path, axis=1)
missing_mask = EPOC_SESSIONS['resolved_file_path'].isna()
print('EPOC sessions:', len(EPOC_SESSIONS))
print('files resolved:', int((~missing_mask).sum()))
print('files missing :', int(missing_mask.sum()))

if missing_mask.any():
    print('\nFirst missing filenames:')
    print(EPOC_SESSIONS.loc[missing_mask, 'file_name'].head(20).to_string(index=False))

if not (~missing_mask).any():
    raise FileNotFoundError(
        'No EPOC CSV files were resolved. Put the BALLADEER ADHD DATASET folder in '
        'Google Drive and rerun this cell.'
    )

# Use the resolved path for all downstream feature extraction.
EPOC_SESSIONS['file_path'] = EPOC_SESSIONS['resolved_file_path']

# ============================================================
# Feature extraction: 1-s windows -> 14 channels x 5 bands = 70
# ============================================================

def safe_cache_name(session_id):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(session_id))


def features_for_session(sess):
    key = safe_cache_name(sess['session_id']) + '__' + str(EPOC_SR) + '__5bands'
    cache_path = os.path.join(CACHE_DIR, key + '.npz')
    if os.path.exists(cache_path):
        z = np.load(cache_path, allow_pickle=True)
        return z['features'], dict(z['checks'].item())

    info = read_epoc_session(sess['file_path'], expected_rate=EPOC_SR,
                             channels=EPOC_CHANNELS)
    windows = make_windows(info['sig'], info['valid'], fs=EPOC_SR,
                           window_samples=SAMPLES_PER_WINDOW)
    features = window_band_features(windows, fs=EPOC_SR, bands=BANDS)

    if features.ndim != 2 or features.shape[1] != 70:
        raise ValueError(f'Expected (n_windows, 70), got {features.shape}')

    checks = {
        'n_rows': info['n_rows'],
        'n_dropped': info['n_dropped'],
        'declared_rate': info['declared_rate'],
        'rate_ok': info['rate_ok'],
        'order_ok': info['order_ok'],
        'timestamp_ok': info['timestamp_ok'],
        'n_windows': len(features),
    }
    np.savez_compressed(cache_path, features=features, checks=checks)
    return features, checks

parse_fails, parse_issues = [], []
session_features = {}

for _, sess in EPOC_SESSIONS.iterrows():
    try:
        feats, checks = features_for_session(sess)
    except Exception as exc:
        parse_fails.append(sess['session_id'])
        parse_issues.append(f"{sess['session_id']}: {repr(exc)}")
        continue

    if len(feats) == 0:
        parse_issues.append(f"{sess['session_id']}: zero complete 1-s windows")
        continue

    session_features[sess['session_id']] = feats
    if checks['n_dropped'] or not checks['order_ok'] or not checks['timestamp_ok']:
        parse_issues.append(
            f"{sess['session_id']}: dropped={checks['n_dropped']} "
            f"order_ok={checks['order_ok']} timestamp_ok={checks['timestamp_ok']}"
        )

print('\nsessions parsed OK:', len(session_features), '| failures:', len(parse_fails))
if session_features:
    example_sid = next(iter(session_features))
    print('example key:', example_sid)
    print('example feature shape:', session_features[example_sid].shape)
    print('feature count:', session_features[example_sid].shape[1])
print('\nFirst issues:')
for issue in parse_issues[:20]:
    print(' ', issue)


In [ ]:
# ============================================================
# Assemble temporally ordered participant sequences
# ============================================================

# Normalize labels before matching.
label_map = (
    labels[['participant_id', 'diagnosis']]
    .drop_duplicates('participant_id')
    .copy()
)
label_map['diagnosis_norm'] = label_map['diagnosis'].astype(str).str.strip().str.lower()
label_dict = dict(zip(label_map['participant_id'], label_map['diagnosis_norm']))

part_diag = {
    pid: label_dict.get(pid, 'unknown')
    for pid in EPOC_SESSIONS['participant_id'].unique()
}

usable = {
    pid for pid, d in part_diag.items()
    if d in ('adhd', 'control')
}

print('Total EPOC participants :', EPOC_SESSIONS['participant_id'].nunique())
print('Usable participants     :', len(usable))
print('ADHD participants       :', sum(part_diag[p] == 'adhd' for p in usable))
print('Control participants    :', sum(part_diag[p] == 'control' for p in usable))

rowsX, rowsY, rowsP, rowsS = [], [], [], []
n_windows_part = {}
n_sequences_part = {}

for pid in sorted(usable):
    psessions = EPOC_SESSIONS[
        EPOC_SESSIONS['participant_id'] == pid
    ].copy()
    # session_id contains the session timestamp; use it only for deterministic ordering.
    psessions = psessions.sort_values(['session_id'])

    total_w = 0
    total_seq = 0

    for _, sess in psessions.iterrows():
        sid = sess['session_id']
        feats = session_features.get(sid)
        if feats is None:
            continue

        n_w = len(feats)
        total_w += n_w
        T = SEQUENCE_LEN

        if n_w < T:
            continue

        for start in range(0, n_w - T + 1, SEQUENCE_STRIDE):
            rowsX.append(feats[start:start + T])
            rowsY.append(1 if part_diag[pid] == 'adhd' else 0)
            rowsP.append(pid)
            rowsS.append(sid)
            total_seq += 1

    n_windows_part[pid] = total_w
    n_sequences_part[pid] = total_seq

if not rowsX:
    raise ValueError(
        'No sequences were created. Check session_features, SEQUENCE_LEN, '
        'and the resolved EPOC files.'
    )

X_seq = np.asarray(rowsX, dtype=np.float32)
y_seq = np.asarray(rowsY, dtype=np.int64)
part_seq = np.asarray(rowsP)
session_seq = np.asarray(rowsS)

if X_seq.ndim != 3 or X_seq.shape[2] != 70:
    raise ValueError(f'Expected X_seq shape (N, T, 70), got {X_seq.shape}')

parts_sorted = sorted(set(part_seq))
code_of = {pid: i for i, pid in enumerate(parts_sorted)}
groups_seq = np.asarray([code_of[p] for p in part_seq], dtype=np.int64)

window_counts = list(n_windows_part.values())
sequence_counts = list(n_sequences_part.values())

print('\n' + '=' * 60)
print('FINAL SEQUENCE DATASET')
print('=' * 60)
print('Sequence length           :', SEQUENCE_LEN, 'seconds')
print('Sequence stride           :', SEQUENCE_STRIDE, 'windows')
print('Total 1-s windows         :', int(sum(window_counts)))
print('Total sequences           :', len(X_seq))
print('X shape                   :', X_seq.shape)
print('y shape                   :', y_seq.shape)
print('Participants              :', len(parts_sorted))
print('Unique participant groups :', len(np.unique(groups_seq)))
print('Sequences/participant     : min=%d max=%d mean=%.1f' % (
    min(sequence_counts), max(sequence_counts), np.mean(sequence_counts)))
print('ADHD sequences            :', int((y_seq == 1).sum()))
print('Control sequences         :', int((y_seq == 0).sum()))


## 7. Baseline model — CNN → LSTM → Dense → Sigmoid

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

class ConvLSTM(nn.Module):
    """CNN over timesteps (local temporal patterns), then LSTM (temporal
    dependencies), then a dense projection to a single logistic output."""
    def __init__(self, n_features, conv_filters=32, conv_kernel=5,
                 lstm_hidden=32, dropout=0.3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_features, conv_filters, kernel_size=conv_kernel,
                      padding=conv_kernel // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.lstm = nn.LSTM(conv_filters, lstm_hidden, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(lstm_hidden, 1)

    def forward(self, x):
        x = x.transpose(1, 2)            # (N, F, T)
        x = self.conv(x).transpose(1, 2) # (N, T, C)
        out, _ = self.lstm(x)            # (N, T, H)
        return self.head(self.dropout(out[:, -1, :])).squeeze(-1), None

m = ConvLSTM(n_features=X_seq.shape[2])
n_params = sum(p.numel() for p in m.parameters())
print(m)
print("param count:", n_params)

## 8. Proposed model — CNN → LSTM → (temporal) Attention → Dense → Sigmoid

**Attention operates over the LSTM's temporal outputs.** The learned weights say which
time positions the **model** found informative for classification - this is a property
of the trained model, NOT evidence of biological importance.

In [ ]:
class TemporalAttention(nn.Module):
    """Additive (Bahdanau-style) attention over the LSTM temporal outputs."""
    def __init__(self, hidden):
        super().__init__()
        self.W = nn.Linear(hidden, hidden)
        self.v = nn.Linear(hidden, 1)

    def forward(self, h):
        e = self.v(torch.tanh(self.W(h))).squeeze(-1)   # (N, T)
        a = torch.softmax(e, dim=-1)                    # (N, T) attention weights
        ctx = torch.sum(a.unsqueeze(-1) * h, dim=1)     # (N, H)
        return ctx, a

class ConvLSTMAttention(nn.Module):
    def __init__(self, n_features, conv_filters=32, conv_kernel=5,
                 lstm_hidden=32, dropout=0.3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_features, conv_filters, kernel_size=conv_kernel,
                      padding=conv_kernel // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.lstm = nn.LSTM(conv_filters, lstm_hidden, batch_first=True)
        self.attn = TemporalAttention(lstm_hidden)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(lstm_hidden, 1)

    def forward(self, x, return_attention=False):
        conv = self.conv(x.transpose(1, 2)).transpose(1, 2)
        out, _ = self.lstm(conv)           # (N, T, H)
        ctx, attn = self.attn(out)
        return self.head(self.dropout(ctx)).squeeze(-1), attn

m2 = ConvLSTMAttention(n_features=X_seq.shape[2])
print(m2)
print("param count:", sum(p.numel() for p in m2.parameters()))
print("attention weights are returned by forward(return_attention=True) for interpretation.")

## 9. Training (identical protocol, identical folds)

In [ ]:
# Shared, leakage-free training protocol
from sklearn.preprocessing import StandardScaler

EPOCHS = 60
BATCH = 128
PATIENCE = 8
LR_INIT = 1e-3


def make_dataloaders(Xtr, ytr, Xva, yva, Xte, yte, bs=BATCH):
    t = torch.utils.data.TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr))
    v = torch.utils.data.TensorDataset(torch.from_numpy(Xva), torch.from_numpy(yva))
    e = torch.utils.data.TensorDataset(torch.from_numpy(Xte), torch.from_numpy(yte))
    return (
        torch.utils.data.DataLoader(t, batch_size=bs, shuffle=True),
        torch.utils.data.DataLoader(v, batch_size=bs, shuffle=False),
        torch.utils.data.DataLoader(e, batch_size=bs, shuffle=False),
    )


def evaluate(model, loader, device=DEVICE):
    model.eval()
    ps, ys, atw = [], [], []
    lossf = nn.BCEWithLogitsLoss(reduction='sum')
    total_loss = 0.0
    total_n = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb_float = yb.float().to(device)
            logits, attn = model(xb)
            total_loss += lossf(logits, yb_float).item()
            total_n += len(yb)
            ps.append(torch.sigmoid(logits).cpu().numpy())
            ys.append(yb.numpy())
            if attn is not None:
                atw.append(attn.cpu().numpy())

    p = np.concatenate(ps)
    y = np.concatenate(ys)
    mean_loss = total_loss / max(total_n, 1)
    attention = np.concatenate(atw) if atw else None
    return p, y, attention, mean_loss


def train_fold(model_cls, Xtr, ytr, Xva, yva, Xte, yte, seed):
    torch.manual_seed(seed)
    model = model_cls(n_features=Xtr.shape[2]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR_INIT, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='min', patience=3, factor=0.5
    )
    lossf = nn.BCEWithLogitsLoss()
    tr_dl, va_dl, te_dl = make_dataloaders(Xtr, ytr, Xva, yva, Xte, yte)

    best_val = float('inf')
    best_state = None
    best_epoch = 0
    wait = 0

    for ep in range(1, EPOCHS + 1):
        model.train()
        for xb, yb in tr_dl:
            xb = xb.to(DEVICE)
            yb = yb.float().to(DEVICE)
            opt.zero_grad()
            logits, _ = model(xb)
            loss = lossf(logits, yb)
            loss.backward()
            opt.step()

        _, _, _, val_loss = evaluate(model, va_dl)
        sched.step(val_loss)

        if val_loss < best_val - 1e-4:
            best_val = val_loss
            best_epoch = ep
            wait = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if wait >= PATIENCE:
                break

    if best_state is None:
        raise RuntimeError('Training never produced a valid checkpoint.')

    model.load_state_dict(best_state)
    pt, yt, atw, _ = evaluate(model, te_dl)
    return model, best_epoch, best_val, pt, yt, atw


In [ ]:
# ---- fixed participant-level folds (same for both models) ----
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
FOLDS = []   # list of (tr_idx, te_idx)
for tr, te in sgkf.split(X_seq, y_seq, groups_seq):
    FOLDS.append((tr, te))
print("fold sizes (sequences):")
for k, (tr, te) in enumerate(FOLDS):
    print(f"  fold {k}: train={len(tr)} test={len(te)} "
          f"| test ADHD={int((y_seq[te]==1).sum())} Control={int((y_seq[te]==0).sum())}")

# participant->fold table persisted for transparency
rows = []
for k, (tr, te) in enumerate(FOLDS):
    for i in te:
        rows.append({"fold": k, "participant_id": part_seq[i],
                     "diagnosis": "ADHD" if y_seq[i] == 1 else "Control"})
fold_df = pd.DataFrame(rows).drop_duplicates()
fold_csv = os.path.join(PROJECT, "results", "fold_assignments.csv")
os.makedirs(os.path.dirname(fold_csv), exist_ok=True)
fold_df.to_csv(fold_csv, index=False)
print("fold assignments saved:", fold_csv)

**Leakage-free normalization**: inside each fold a `StandardScaler` is fit on the
**training participants only** (all their sequences, flattened across time) and applied
to validation/test. Validation set is used only for early stopping / LR scheduling;
the test set is touched only for the final per-fold report. **No test tuning.**

In [ ]:
def run_protocol(model_name, model_cls):
    per_fold = []
    for k, (tr, te) in enumerate(FOLDS):
        # Validation participants are drawn only from the training participants.
        tr_groups = groups_seq[tr]
        gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED + k)
        tr_i, va_i = next(gss.split(np.zeros(len(tr)), y_seq[tr], groups=tr_groups))
        tr_all = tr[tr_i]
        va = tr[va_i]

        # Fit normalization on training participants only.
        sc = StandardScaler().fit(X_seq[tr_all].reshape(-1, X_seq.shape[2]))
        Xtr = sc.transform(X_seq[tr_all].reshape(-1, X_seq.shape[2])).reshape(X_seq[tr_all].shape)
        Xva = sc.transform(X_seq[va].reshape(-1, X_seq.shape[2])).reshape(X_seq[va].shape)
        Xte = sc.transform(X_seq[te].reshape(-1, X_seq.shape[2])).reshape(X_seq[te].shape)

        model, best_ep, best_val, p, yt, atw = train_fold(
            model_cls,
            Xtr.astype('float32'), y_seq[tr_all],
            Xva.astype('float32'), y_seq[va],
            Xte.astype('float32'), y_seq[te],
            seed=SEED + k,
        )

        print(
            f'[{model_name}] fold {k} done in {best_ep} epochs '
            f'(val_loss={best_val:.4f}) test_n={len(yt)}'
        )

        per_fold.append({
            'fold': k,
            'model_name': model_name,
            'p': p,
            'y': yt,
            'atw': atw,
            'pid': part_seq[te],
            'session': session_seq[te],
            'epochs': best_ep,
            'scaler': sc,
            'model': model,
            '_te_idx': te,
        })

    return per_fold

print('training protocol identical for both models (same folds, seeds, and scaler policy).')


## 10. Evaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve,
    precision_recall_curve, average_precision_score, brier_score_loss
)
from sklearn.calibration import calibration_curve
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def compute_metrics(y, p):
    pred = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else np.nan
    spec = tn / (tn + fp) if (tn + fp) else np.nan
    return {
        'accuracy': accuracy_score(y, pred),
        'precision': precision_score(y, pred, zero_division=0),
        'recall': recall_score(y, pred, zero_division=0),
        'f1': f1_score(y, pred, zero_division=0),
        'roc_auc': roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan,
        'sensitivity': sens,
        'specificity': spec,
        'brier': brier_score_loss(y, p),
        'n': len(y),
    }


In [ ]:
run = run_protocol("CNN_LSTM", ConvLSTM)
run2 = run_protocol("CNN_LSTM_Attention", ConvLSTMAttention)
ALL_FOLD_RESULTS = {"CNN_LSTM": run, "CNN_LSTM_Attention": run2}

metric_names = ["accuracy", "precision", "recall", "f1", "roc_auc",
                "sensitivity", "specificity", "brier"]
summary_rows = []
for mname in ["CNN_LSTM", "CNN_LSTM_Attention"]:
    for r in ALL_FOLD_RESULTS[mname]:
        met = compute_metrics(r["y"], r["p"])
        row = {"model": mname, "fold": r["fold"], "n_test": met["n"]}
        row.update({k: met[k] for k in metric_names})
        summary_rows.append(row)
fold_metrics = pd.DataFrame(summary_rows)
print(fold_metrics.round(3).to_string(index=False))

agg = fold_metrics.groupby("model")[metric_names].agg(["mean", "std"]).round(4)
print("\nmean +/- std per model (test folds):")
print(agg.to_string())

In [ ]:
# participant-level aggregation: majority prediction per participant (NEVER window-level-as-subject)
part_summary_rows = []
for mname in ["CNN_LSTM", "CNN_LSTM_Attention"]:
    acc = {}
    for r in ALL_FOLD_RESULTS[mname]:
        pred = (r["p"] >= 0.5).astype(int)
        for pid, py, pl in zip(r["pid"], pred, r["y"]):
            acc.setdefault(pid, []).append((py, pl))
    for pid, lst in acc.items():
        maj = int(np.mean([x[0] for x in lst]) >= 0.5)
        true = int(np.mean([x[1] for x in lst]) >= 0.5)   # participant is single-label
        part_summary_rows.append({"model": mname, "participant_id": pid,
                                  "true": true, "pred": maj})
part_df = pd.DataFrame(part_summary_rows)
pm = part_df.groupby(["model", "true", "pred"]).size().unstack(fill_value=0)
print("participant-level confusion, both models:")
print(pm.to_string())
print("\nparticipant-level accuracy per model:")
for mname in ["CNN_LSTM", "CNN_LSTM_Attention"]:
    g = part_df[part_df.model == mname]
    print(f"  {mname:22s}: {np.mean(g.true == g.pred):.4f} "
          f"(n participants = {len(g)}, n ADHD = {(g.true == 1).sum()}, n Control = {(g.true == 0).sum()})")

**Confusion matrices / ROC / PR / calibration curves are plotted and saved below** —
per fold and pooled across folds.

In [ ]:
PLOTS_DIR = os.path.join(PROJECT, 'plots')
os.makedirs(PLOTS_DIR, exist_ok=True)

def pooled_y_p(mname):
    Y = np.concatenate([r['y'] for r in ALL_FOLD_RESULTS[mname]])
    P = np.concatenate([r['p'] for r in ALL_FOLD_RESULTS[mname]])
    return Y, P

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for mname in ['CNN_LSTM', 'CNN_LSTM_Attention']:
    Y, P = pooled_y_p(mname)
    fpr, tpr, _ = roc_curve(Y, P)
    prec, rec, _ = precision_recall_curve(Y, P)
    axes[0].plot(fpr, tpr, label=f'{mname} (AUC={roc_auc_score(Y, P):.3f})')
    axes[1].plot(rec, prec, label=f'{mname} (AP={average_precision_score(Y, P):.3f})')
    prob_true, prob_pred = calibration_curve(Y, P, n_bins=10, strategy='quantile')
    axes[2].plot(prob_pred, prob_true, marker='o', label=f'{mname} (Brier={brier_score_loss(Y, P):.3f})')

axes[0].plot([0, 1], [0, 1], '--', lw=1)
axes[0].set(xlabel='False Positive Rate', ylabel='True Positive Rate', title='ROC')
axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall')
axes[2].plot([0, 1], [0, 1], '--', lw=1)
axes[2].set(xlabel='Mean predicted probability', ylabel='Fraction positive', title='Calibration')
for ax in axes:
    ax.legend(loc='best', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'roc_pr_calibration.png'), dpi=130)
plt.show()
print('saved plots/roc_pr_calibration.png')


## 11. 95% bootstrap confidence intervals (participant-level)

In [ ]:
def bootstrap_participant_ci(mname, metric, n_boot=1000, alpha=0.05):
    # First aggregate all test windows belonging to each participant.
    parts = {}
    for r in ALL_FOLD_RESULTS[mname]:
        for pid, py, pl in zip(r['pid'], r['y'], r['p']):
            parts.setdefault(pid, {'y': int(py), 'p': []})['p'].append(float(pl))

    pids_all = list(parts.keys())
    if len(pids_all) < 2:
        return np.nan, np.nan, np.nan

    y_part = np.asarray([parts[pid]['y'] for pid in pids_all], dtype=int)
    p_part = np.asarray([np.mean(parts[pid]['p']) for pid in pids_all], dtype=float)

    def metric_value(yv, pv):
        if metric == 'roc_auc':
            return roc_auc_score(yv, pv) if len(np.unique(yv)) == 2 else np.nan
        if metric == 'brier':
            return brier_score_loss(yv, pv)
        pred = (pv >= 0.5).astype(int)
        fn = {'accuracy': accuracy_score, 'precision': precision_score,
              'recall': recall_score, 'f1': f1_score}[metric]
        return fn(yv, pred, zero_division=0)

    rng = np.random.default_rng(SEED)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(pids_all), size=len(pids_all))
        v = metric_value(y_part[idx], p_part[idx])
        if np.isfinite(v):
            vals.append(v)

    if not vals:
        return np.nan, np.nan, np.nan

    lo = float(np.percentile(vals, 100 * alpha / 2))
    hi = float(np.percentile(vals, 100 * (1 - alpha / 2)))
    return lo, hi, float(np.mean(vals))

ci_rows = []
for mname in ['CNN_LSTM', 'CNN_LSTM_Attention']:
    for met in ['accuracy', 'f1', 'roc_auc', 'brier']:
        lo, hi, mean = bootstrap_participant_ci(mname, met)
        ci_rows.append({
            'model': mname,
            'metric': met,
            'mean': round(mean, 3) if np.isfinite(mean) else np.nan,
            'ci95_low': round(lo, 3) if np.isfinite(lo) else np.nan,
            'ci95_high': round(hi, 3) if np.isfinite(hi) else np.nan,
        })

ci_df = pd.DataFrame(ci_rows)
print('participant-level 95% bootstrap CIs (1000 resamples):')
print(ci_df.to_string(index=False))


## 12. Attention analysis (model-internal, not neurological)

In [ ]:
import numpy as np
attn_by_model = {}
attn_rows = []
for r in ALL_FOLD_RESULTS["CNN_LSTM_Attention"]:
    if r["atw"] is None:
        continue
    a = r["atw"]                       # (N, T)
    pred = (r["p"] >= 0.5).astype(int)
    correct = (pred == r["y"]).astype(int)
    attn_rows.append({"fold": r["fold"]})
    mean_a = a.mean(axis=0)            # (T,)   avg attention over temporal positions
    mean_a_correct = a[correct == 1].mean(axis=0) if correct.sum() else np.zeros(a.shape[1])
    mean_a_wrong = a[correct == 0].mean(axis=0) if (correct == 0).sum() else np.zeros(a.shape[1])
    attn_by_model[r["fold"]] = {"mean": mean_a, "correct": mean_a_correct, "wrong": mean_a_wrong,
                                "n_correct": int(correct.sum()), "n_wrong": int((correct == 0).sum())}

fig, ax = plt.subplots(figsize=(9, 4))
T = SEQUENCE_LEN
xs = np.arange(T)
folds = sorted(attn_by_model)
for k in folds:
    ax.plot(xs, attn_by_model[k]["mean"], alpha=0.4, lw=1)
ax.set_xlabel("Temporal window index within sequence (oldest -> newest)")
ax.set_ylabel("Mean attention weight")
ax.set_title("Average attention over temporal windows, per fold (CNN-LSTM-Attention)")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "attention_temporal.png"), dpi=130)
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
for k in folds:
    ax.bar(xs, attn_by_model[k]["correct"], alpha=0.4, color="tab:green", label="correct" if k == folds[0] else None)
    ax.bar(xs, attn_by_model[k]["wrong"], alpha=0.4, color="tab:red", label="wrong" if k == folds[0] else None)
ax.set_xlabel("Temporal window index")
ax.set_ylabel("Mean attention weight")
ax.set_title("Attention: correctly vs incorrectly classified windows (fold overlays)")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "attention_correct_v_wrong.png"), dpi=130)
plt.show()
print("attention visualisations saved.")
print("NOTE: attention reflects what the MODEL found informative - it is not evidence of")
print("neurological causality or of any biological importance.")

## 13. Channel contribution analysis (permutation-based)

**Approach (channel-contribution analysis, not "CNN-weight importance").**
For each fold we re-transform that fold's test sequences with the fold's fitted scaler,
shuffle the channel's 5 band columns across the fold's test sequences, re-run the stored
fold model, and record the drop in that fold's ROC-AUC vs its unpermuted predictions.
Mean drop per channel (across folds and permutations) ranks channel contribution. Only
test info from the fold it was bound to is used (no leakage). Run cell below.

In [ ]:
# (materialised) channel permutation using stored fold models + scalers:
import torch

ch_imp = {}
for mname in ["CNN_LSTM", "CNN_LSTM_Attention"]:
    imp = []
    base_auc = np.mean([roc_auc_score(r["y"], r["p"]) for r in ALL_FOLD_RESULTS[mname]])
    for ci, ch in enumerate(EPOC_CHANNELS):
        cols = slice(ci * 5, ci * 5 + 5)
        drops = []
        for r in ALL_FOLD_RESULTS[mname]:
            Xte = r["scaler"].transform(
                X_seq[r["_te_idx"]].reshape(-1, X_seq.shape[2])
            ).reshape(*X_seq[r["_te_idx"]].shape).astype("float32")
            model = r["model"].to(DEVICE); model.eval()
            auc_orig = roc_auc_score(r["y"], r["p"])
            for _ in range(3):
                rng = np.random.default_rng(SEED + ci * 10 + _)
                idx = rng.permutation(Xte.shape[0])
                Xp = Xte.copy()
                Xp[:, :, cols] = Xp[idx][:, :, cols]
                with torch.no_grad():
                    logits, _ = model(torch.from_numpy(Xp).to(DEVICE))
                    pp = torch.sigmoid(logits).cpu().numpy()
                drops.append(auc_orig - roc_auc_score(r["y"], pp))
        imp.append((ch, float(np.mean(drops))))
    ch_imp[mname] = imp
ch_imp_df = pd.DataFrame({"channel": [c for c, _ in ch_imp["CNN_LSTM_Attention"]]})
for mname in ["CNN_LSTM", "CNN_LSTM_Attention"]:
    ch_imp_df[mname] = [v for _, v in ch_imp[mname]]
print("permutation channel importance (mean ROC-AUC drop, higher = more important):")
print(ch_imp_df.sort_values("CNN_LSTM_Attention", ascending=False).round(4).to_string(index=False))
print()
print("-> raw CNN weights are NOT used as channel importance; permutation determines "
      "the drop when a channel's band features are shuffled.")

## 14. Optional ablation (4-band baseline vs 5-band extension, both models)

The main comparison remains **CNN-LSTM vs CNN-LSTM-Attention**. The band ablation is
optional and is **not run by default** (resource saving). Set `RUN_ABLATION = True` to
re-extract features with `Delta + Theta + Alpha + Beta` only and rerun both models with
the same folds/protocol.

In [ ]:
RUN_ABLATION = False
if RUN_ABLATION:
    print("Ablation enabled: rerunning with 4 baseline bands (Gamma dropped).")
    raise SystemExit("Ablation requires re-extraction from the cleaned sessions; "
                     "edit BANDS above to 4 bands and rerun Sections 5-10.")
else:
    print("Optional ablation not run (resource policy). Main comparison is CNN-LSTM "
          "vs CNN-LSTM-Attention on the same folds.")

## 15. Error analysis

In [ ]:
err_rows = []
for r in ALL_FOLD_RESULTS["CNN_LSTM_Attention"]:
    pred = (r["p"] >= 0.5).astype(int)
    for pid, s, y, pr, pv in zip(r["pid"], r["session"], r["y"], pred, r["p"]):
        err_rows.append({"model": "CNN_LSTM_Attention", "participant_id": pid,
                         "session_id": s, "true": y, "pred": pr, "prob": pv,
                         "type": "FP" if (y == 0 and pr == 1) else
                                 "FN" if (y == 1 and pr == 0) else "correct"})
err = pd.DataFrame(err_rows)
print("CNN-LSTM-Attention error breakdown:")
print(err["type"].value_counts().to_string())
print("\nparticipants with >=1 misclassified window:")
print(err[err["type"] != "correct"].groupby("participant_id")["type"].count()
      .sort_values(ascending=False).head(10).to_string())

# per-task performance (Attention model)
def per_task():
    out = {}
    for r in ALL_FOLD_RESULTS["CNN_LSTM_Attention"]:
        pred = (r["p"] >= 0.5).astype(int)
        df = pd.DataFrame({"session_id": r["session"], "y": r["y"], "pred": pred})
        task = EPOC_SESSIONS.set_index("session_id")["task"].to_dict()
        df["task"] = df["session_id"].map(task)
        for t, g in df.groupby("task"):
            out.setdefault(t, []).append(accuracy_score(g["y"], g["pred"]))
    return {t: np.mean(v) for t, v in out.items()}
print("\naccuracy by task (window-level):", per_task())

# per age / gender (metadata supports it)
meta_cols = ["participant_id", "age", "gender", "group"]
pmt = pmeta[meta_cols].drop_duplicates("participant_id")
pmt["gender"] = pmt["gender"].map({"1": "male", "2": "female"})
merged = err.merge(pmt, on="participant_id", how="left")
print("\nmisclassification rate by gender:")
print(merged.groupby("gender")["type"].apply(lambda s: (s != "correct").mean()).to_string())
print("\nmean age correct vs errors:")
print(merged.groupby((merged["type"] != "correct").map({True: "error", False: "correct"}))
      ["age"].mean().to_string())
print("\nDISCLAIMER: these are descriptive correlations of prediction errors - "
      "no medical cause is inferred.")

## 16. Results tables

In [ ]:
final_tab = fold_metrics.groupby("model")[
    ["accuracy", "precision", "recall", "f1", "roc_auc"]].mean().round(3)
final_tab.columns = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
ci_map = {}
for _, r in ci_df.iterrows():
    ci_map[(r["model"], r["metric"])] = (r["ci95_low"], r["ci95_high"])
final_tab["95% CI (acc)"] = [
    f"[{ci_map[(m,'accuracy')][0]:.3f}, {ci_map[(m,'accuracy')][1]:.3f}]" for m in final_tab.index]
final_tab["95% CI (AUC)"] = [
    f"[{ci_map[(m,'roc_auc')][0]:.3f}, {ci_map[(m,'roc_auc')][1]:.3f}]" for m in final_tab.index]
print(final_tab.to_string())

In [ ]:
base = fold_metrics[fold_metrics["model"] == "CNN_LSTM"].set_index("fold")
attn = fold_metrics[fold_metrics["model"] == "CNN_LSTM_Attention"].set_index("fold")
table = pd.DataFrame({
    "Model": ["CNN_LSTM"] * len(base) + ["CNN_LSTM_Attention"] * len(attn),
    "Fold": list(base.index) + list(attn.index),
    "Accuracy": pd.concat([base["accuracy"], attn["accuracy"]]).values,
    "F1": pd.concat([base["f1"], attn["f1"]]).values,
    "ROC-AUC": pd.concat([base["roc_auc"], attn["roc_auc"]]).values,
})
print("same-fold comparison (test folds):")
print(table.round(3).to_string(index=False))

## 17. Saving outputs

In [ ]:
RUN_ID = datetime.now().strftime("run_%Y%m%d_%H%M%S")
OUT_DIRS = {k: os.path.join(PROJECT, k, RUN_ID) for k in ["models", "results", "plots", "predictions"]}
for d in OUT_DIRS.values():
    os.makedirs(d, exist_ok=True)
print("experiment outputs ->", RUN_ID)
print({k: v for k, v in OUT_DIRS.items()})

# models + weights
for mname in ["CNN_LSTM", "CNN_LSTM_Attention"]:
    for r in ALL_FOLD_RESULTS[mname]:
        torch.save(r["model"].state_dict(),
                   os.path.join(OUT_DIRS["models"], f"{mname}_fold{r['fold']}.pt"))

# predictions + per-fold metrics
pred_rows = []
for mname in ["CNN_LSTM", "CNN_LSTM_Attention"]:
    for r in ALL_FOLD_RESULTS[mname]:
        pr = (r["p"] >= 0.5).astype(int)
        for pid, s, y, pv, pp in zip(r["pid"], r["session"], r["y"], r["p"], pr):
            pred_rows.append({"model": mname, "fold": r["fold"], "participant_id": pid,
                              "session_id": s, "true": y, "prob": pv, "pred": pp})
pd.DataFrame(pred_rows).to_csv(
    os.path.join(OUT_DIRS["predictions"], "predictions.csv"), index=False)

fold_metrics.to_csv(os.path.join(OUT_DIRS["results"], "fold_metrics.csv"), index=False)
final_tab.to_csv(os.path.join(OUT_DIRS["results"], "final_metrics.csv"))
ci_df.to_csv(os.path.join(OUT_DIRS["results"], "confidence_intervals.csv"), index=False)
fold_df.to_csv(os.path.join(OUT_DIRS["results"], "fold_assignments.csv"), index=False)

# scalers + attention
for mname in ["CNN_LSTM", "CNN_LSTM_Attention"]:
    for r in ALL_FOLD_RESULTS[mname]:
        joblib.dump(r["scaler"],
                    os.path.join(OUT_DIRS["models"], f"{mname}_fold{r['fold']}_scaler.joblib"))
attn_pack = {k: v for k, v in attn_by_model.items()}
np.savez(os.path.join(OUT_DIRS["results"], "attention_weights.npz"),
         **{f"fold{k}": v for k, v in attn_pack.items()},
         allow_pickle=True)

# plots already saved to plots/; copy a pointer
shared_plots = []
for fn_ in ["roc_pr_calibration.png", "attention_temporal.png", "attention_correct_v_wrong.png"]:
    src = os.path.join(PROJECT, "plots", fn_)
    if os.path.exists(src):
        with open(os.path.join(OUT_DIRS["plots"], "pointer_" + fn_ + ".txt"), "w") as fh:
            fh.write(src)
        shared_plots.append(fn_)
print("shared plots retained:", shared_plots)

summary_json = {
    "run_id": RUN_ID, "n_participants": len(parts_sorted),
    "n_sequences": int(len(X_seq)), "window_sec": WINDOW_SEC,
    "sequence_len": SEQUENCE_LEN, "bands": BAND_NAMES,
    "sampling_rate": EPOC_SR, "evaluation": "StratifiedGroupKFold(5)",
    "final_metrics": final_tab.to_dict(),
}
with open(os.path.join(OUT_DIRS["results"], "run_summary.json"), "w", encoding="utf-8") as fh:
    json.dump(summary_json, fh, indent=2, default=str)
print("run summary written. Nothing is overwritten: every run has a fresh RUN_ID.")

## 18. Final report

In [ ]:
print('========================= FINAL REPORT =========================')
print(f'participants trained on     : {len(parts_sorted)} (EPOC 128 Hz, ADHD+Control only)')
print(f'ADHD / Control              : {int((y_seq==1).sum())} / {int((y_seq==0).sum())} sequences')
print(f'unique participants         : {len(parts_sorted)}')
print(f'EEG channels                : {", ".join(EPOC_CHANNELS)}')
print(f'sampling rate               : {EPOC_SR} Hz (declared eeg_128 subset)')
print(f'window size                 : {WINDOW_SEC} s ({SAMPLES_PER_WINDOW} samples/window)')
print(f'frequency bands             : {", ".join(BAND_NAMES)} (Gamma = extension of 4-band baseline)')
print(f'features per timestep       : {len(EPOC_CHANNELS)} channels x {len(BAND_NAMES)} bands = {len(EPOC_CHANNELS)*len(BAND_NAMES)}')
print(f'sequence length             : {SEQUENCE_LEN} one-second timesteps')
print(f'number of sequences         : {len(X_seq)}  tensor shape {tuple(X_seq.shape)}')
print(f'evaluation strategy         : StratifiedGroupKFold(5), groups=participant_id')
print(f'CNN-LSTM                    : {fold_metrics[fold_metrics.model=="CNN_LSTM"][['accuracy','f1']].mean().round(3).to_dict()}')
print(f'CNN-LSTM-Attention          : {fold_metrics[fold_metrics.model=="CNN_LSTM_Attention"][['accuracy','f1']].mean().round(3).to_dict()}')
print('95% bootstrap CI (participant-level):')
for _, rr in ci_df[ci_df.metric=='roc_auc'].iterrows():
    print(f"   {rr['model']:<22s} mean={rr['mean']:.3f} CI=[{rr['ci95_low']:.3f}, {rr['ci95_high']:.3f}]")
print('attention findings          : model-internal temporal weights, not biological evidence')
print('channel contribution        : permutation-based AUC drop, not raw CNN weights')
print('limitations                 : only verified 128-Hz EPOC sessions used; other sources excluded from this model')
print('================================================================')
